# Validation-Retry Loops for Structured Extraction

When Claude's structured output fails validation, two very different root causes require different responses:

| Failure type | Root cause | Right response |
|---|---|---|
| **Formatting Error** | Data exists in the source; Claude used the wrong format | Append the validation error and retry — Claude self-corrects |
| **Missing Information** | Data does not exist in source; `required` forces fabrication | Fix the schema — make the field nullable; retry only produces different fake values |

| Stage | Concept | Key technique |
|---|---|---|
| 1. Hallucination baseline | `required` creates fabrication pressure when fields are absent | See the failure mode |
| 2. Validation function | Detect format errors programmatically | Custom validator |
| 3. Formatting Error retry | Append error message and retry — converges in 1-2 attempts | Retry loop |
| 4. Missing Information retry | 3 retries produce 3 different fake values | Divergence diagnosis |
| 5. Root fix | Nullable schema stops fabrication at the source | nullable union type |

> **Terminology note:** This notebook uses *Formatting Error* and *Missing Information* as operational labels.
> Mapped to standard AI engineering terms: Formatting Error = **Format Error**; Missing Information is one subtype of **Semantic Error** (output that is format-valid but semantically wrong — hallucinated values that pass schema validation undetected).
> A second Semantic Error subtype — *Misinterpretation* (field exists in source but is misread) — requires confidence scoring and human review to detect, not schema changes.
>
> → For full **Semantic Error** coverage (both subtypes, confidence scoring, Human Review Queue routing): [`semantic_error_detection.ipynb`](semantic_error_detection.ipynb)

In [ ]:
import Anthropic from 'npm:@anthropic-ai/sdk';
import { load } from 'https://deno.land/std@0.220.0/dotenv/mod.ts';

await load({ envPath: '../.env', export: true });

const client = new Anthropic({ apiKey: Deno.env.get('ANTHROPIC_API_KEY') ?? '' });
const MODEL = 'claude-sonnet-4-6';

console.log('Setup complete. Model:', MODEL);

## Stage 1: Hallucination Under `required` Pressure

The simplest extraction schema lists every field as `required`. This works when the source document
contains all the data — but when a field is **absent**, `required` creates pressure on Claude to
fill in a plausible-sounding value rather than admit the field is missing.

The schema below requires five fields. The input document contains only three:
`invoice_number`, `issue_date`, and `amount`. The fields `payment_terms` and `assigned_rep`
do not appear anywhere in the text.

In [ ]:
const invoiceToolAllRequired: Anthropic.Tool = {
  name: 'extract_invoice',
  description: 'Extract key fields from an invoice document.',
  input_schema: {
    type: 'object',
    properties: {
      invoice_number: { type: 'string', description: 'Invoice identifier' },
      issue_date:     { type: 'string', description: 'Invoice issue date' },
      amount:         { type: 'number', description: 'Total amount due in USD' },
      payment_terms:  { type: 'string', description: 'Payment terms (e.g. Net 30)' },
      assigned_rep:   { type: 'string', description: 'Assigned sales representative ID' },
    },
    required: ['invoice_number', 'issue_date', 'amount', 'payment_terms', 'assigned_rep'],
  },
};

// payment_terms and assigned_rep do not appear in this text
const invoiceText =
  'Invoice INV-2026-0847, dated March 5th, 2026. ' +
  'Total amount due: $4,250. Please remit payment at your earliest convenience.';

const stage1Response = await client.messages.create({
  model: MODEL,
  max_tokens: 512,
  tools: [invoiceToolAllRequired],
  tool_choice: { type: 'tool', name: 'extract_invoice' },
  messages: [{ role: 'user', content: invoiceText }],
});

const stage1Block = stage1Response.content.find(b => b.type === 'tool_use');
if (stage1Block?.type === 'tool_use') {
  console.log('Stage 1 — all fields required, input missing two fields:');
  console.log(JSON.stringify(stage1Block.input, null, 2));
  console.log('');
  console.log('(payment_terms and assigned_rep were fabricated — not present in source text)');
}

## Stage 2: A Validation Function

Before retrying, you need a function that inspects the output and returns a specific error message.
That error message is appended to the prompt — Claude reads it and self-corrects.

A useful validation error has three parts:
1. **Which field** failed (`issue_date`)
2. **What was received** (`March 5th, 2026`)
3. **What was expected** (YYYY-MM-DD format)

Vague errors (`invalid output`) do not help Claude converge.
Precise errors (`issue_date must be YYYY-MM-DD, received March 5th, 2026`) converge in 1-2 retries.

In [ ]:
type ValidationResult = { valid: true } | { valid: false; error: string };

function validateDateFormat(data: Record<string, unknown>): ValidationResult {
  const dateStr = data['issue_date'] as string | undefined;
  if (typeof dateStr === 'string' && !/^\d{4}-\d{2}-\d{2}$/.test(dateStr)) {
    return {
      valid: false,
      error: 'issue_date must be YYYY-MM-DD format (e.g. 2026-03-05), received: ' + dateStr,
    };
  }
  return { valid: true };
}

console.log('Validation function defined.');
console.log('Test — prose date:  ', validateDateFormat({ issue_date: 'March 5th, 2026' }));
console.log('Test — ISO date:    ', validateDateFormat({ issue_date: '2026-03-05' }));

## Stage 3: Formatting Error — Retry Converges

A **Formatting Error** is when the data exists in the source document but Claude expressed it in
the wrong format. The invoice clearly states the date — Claude just echoed the prose format
from the input text because the schema description did not specify ISO format.

The retry loop:
1. Call Claude → validate output
2. If validation fails, **append the exact error** to the conversation and call again
3. Repeat up to `maxAttempts` times

Claude sees the specific error and corrects the format on the next attempt. This converges
because the underlying data exists — Claude only needs to reformat it, not invent it.

In [ ]:
// Schema without a date format rule in the description
// Claude will echo the prose date format from the input text on the first attempt
const invoiceToolNoFormat: Anthropic.Tool = {
  name: 'extract_invoice',
  description: 'Extract key fields from an invoice document.',
  input_schema: {
    type: 'object',
    properties: {
      invoice_number: { type: 'string', description: 'Invoice identifier' },
      issue_date:     { type: 'string', description: 'Invoice issue date' },
      amount:         { type: 'number', description: 'Total amount due in USD' },
    },
    required: ['invoice_number', 'issue_date', 'amount'],
  },
};

async function retryWithError(
  text: string,
  tool: Anthropic.Tool,
  validate: (data: Record<string, unknown>) => ValidationResult,
  maxAttempts = 3,
): Promise<void> {
  let messages: Anthropic.MessageParam[] = [{ role: 'user', content: text }];

  for (let attempt = 1; attempt <= maxAttempts; attempt++) {
    const response = await client.messages.create({
      model: MODEL,
      max_tokens: 512,
      tools: [tool],
      tool_choice: { type: 'tool', name: tool.name },
      messages,
    });

    const block = response.content.find(b => b.type === 'tool_use');
    if (!block || block.type !== 'tool_use') break;

    const data = block.input as Record<string, unknown>;
    const result = validate(data);

    if (result.valid) {
      console.log('Attempt ' + attempt + ': VALID');
      console.log(JSON.stringify(data, null, 2));
      return;
    }

    console.log('Attempt ' + attempt + ': INVALID — ' + result.error);

    messages = [
      ...messages,
      { role: 'assistant', content: response.content } as Anthropic.MessageParam,
      { role: 'user', content: 'Validation failed: ' + result.error + '. Please correct the output.' },
    ];
  }

  console.log('Max attempts reached without convergence.');
}

console.log('--- Formatting Error retry (date without format rule) ---');
await retryWithError(invoiceText, invoiceToolNoFormat, validateDateFormat);

## Stage 4: Missing Information — Retry Diverges

**Missing Information** is the opposite case: the field is `required` but the data simply does not
exist in the source document. Claude cannot correct its output by reformatting — there is nothing
to reformat. Instead, each retry produces a **different fabricated value**, since Claude is guessing.

The feedback email below contains the customer name and issue type, but:
- No `resolution_date` — the issue is still unresolved
- No `agent_id` — no agent has been assigned yet

Running the same call three times independently shows diverging outputs — a reliable signal
that you are dealing with Missing Information, not a Formatting Error.

In [ ]:
const feedbackToolAllRequired: Anthropic.Tool = {
  name: 'extract_feedback',
  description: 'Extract structured data from a customer feedback email.',
  input_schema: {
    type: 'object',
    properties: {
      customer_name:   { type: 'string', description: 'Customer full name' },
      issue_type: {
        type: 'string',
        enum: ['billing', 'shipping', 'technical', 'other'],
        description: 'Category of the issue',
      },
      resolution_date: { type: 'string', description: 'Issue resolution date in YYYY-MM-DD' },
      agent_id:        { type: 'string', description: 'Assigned support agent ID' },
    },
    required: ['customer_name', 'issue_type', 'resolution_date', 'agent_id'],
  },
};

// resolution_date and agent_id are genuinely absent from this email
const feedbackEmail =
  'Hi, my name is Sarah Chen. I ordered a laptop last week (order #ORD-9921) ' +
  'and it arrived damaged. The screen has a visible crack and the packaging was crushed. ' +
  'I am very disappointed and need this resolved urgently.';

console.log('--- Missing Information: 3 independent attempts ---');
console.log('');

for (let i = 1; i <= 3; i++) {
  const response = await client.messages.create({
    model: MODEL,
    max_tokens: 512,
    tools: [feedbackToolAllRequired],
    tool_choice: { type: 'tool', name: 'extract_feedback' },
    messages: [{ role: 'user', content: feedbackEmail }],
  });
  const block = response.content.find(b => b.type === 'tool_use');
  if (block?.type === 'tool_use') {
    const data = block.input as Record<string, unknown>;
    console.log(
      'Attempt ' + i + ':  resolution_date=' + data['resolution_date'] +
      '  agent_id=' + data['agent_id']
    );
  }
}

console.log('');
console.log('Diagnosis: different values across attempts → Missing Information.');
console.log('Retrying will not help — there is no resolution date or agent ID in the email.');

## Diagnosing the Failure Type

After 2-3 failed retries, compare the outputs for the failing field:

| Pattern across retries | Diagnosis | Action |
|---|---|---|
| Same wrong value every time | Formatting Error | Keep retrying, or improve the error message |
| Different value on every retry | Missing Information | Stop retrying — fix the schema |
| Converges in 1-2 attempts | Formatting Error (resolved) | Done |

**The silent risk of misdiagnosis**: a fabricated value will eventually *pass* format validation
(e.g., a random `YYYY-MM-DD` date matches the regex check) and enter the production database
undetected. The retry loop filters format — it cannot detect that a correctly formatted value
is still hallucinated.

If `resolution_date` keeps changing across retries (`2026-06-10` → `2026-06-01` → `2026-05-28`),
one of those values will eventually match the format check and silently pollute downstream data.

## Stage 5: Root Fix — Nullable Schema

The correct fix for Missing Information is a schema change, not a better retry strategy.
Fields that may legitimately be absent should:
1. Be removed from `required`
2. Have their type changed to a nullable union: `type: ['string', 'null']`
3. Have a system prompt rule: `"If any field cannot be found, return null"`

Both schema and prompt layers are needed together:
- `type: ['string', 'null']` makes `null` a **legal type** — but does not tell Claude *when* to use it
- The prompt rule tells Claude *when* to return null — but cannot make `null` type-safe without the schema allowing it

> For a full walkthrough of nullable schema design and format normalization patterns,
> see [`null_handling_and_normalization.ipynb`](null_handling_and_normalization.ipynb) in this folder.

In [ ]:
const feedbackToolNullable: Anthropic.Tool = {
  name: 'extract_feedback',
  description: 'Extract structured data from a customer feedback email.',
  input_schema: {
    type: 'object',
    properties: {
      customer_name: { type: 'string', description: 'Customer full name' },
      issue_type: {
        type: 'string',
        enum: ['billing', 'shipping', 'technical', 'other'],
        description: 'Category of the issue',
      },
      resolution_date: {
        type: ['string', 'null'],
        description: 'Issue resolution date in YYYY-MM-DD, or null if not yet resolved',
      },
      agent_id: {
        type: ['string', 'null'],
        description: 'Assigned support agent ID, or null if no agent has been assigned',
      },
    },
    required: ['customer_name', 'issue_type'],
  },
};

const NULL_RULE =
  'If any field cannot be found in the text, return null for that field. ' +
  'Do not guess or fabricate values.';

const fixedResponse = await client.messages.create({
  model: MODEL,
  max_tokens: 512,
  system: NULL_RULE,
  tools: [feedbackToolNullable],
  tool_choice: { type: 'tool', name: 'extract_feedback' },
  messages: [{ role: 'user', content: feedbackEmail }],
});

const fixedBlock = fixedResponse.content.find(b => b.type === 'tool_use');
if (fixedBlock?.type === 'tool_use') {
  console.log('Root fix — nullable schema output:');
  console.log(JSON.stringify(fixedBlock.input, null, 2));
  console.log('');
  console.log('(resolution_date: null — not yet resolved; agent_id: null — not yet assigned)');
  console.log('Downstream: if (data.resolution_date === null) → flag for follow-up');
}

## Summary

### When retry helps vs. when it does not

| | Formatting Error | Missing Information |
|---|---|---|
| **What happened** | Data exists; Claude used the wrong format | Data does not exist in the source document |
| **Retry pattern** | Same wrong format each time | Different fake value on every attempt |
| **Retry result** | Converges in 1-2 attempts | Produces diverging fabricated values |
| **Silent risk** | Retry fixes it cleanly | A randomly-correct format may pass validation undetected |
| **Right action** | Retry with appended error | Fix the schema — use nullable type |

### The retry loop mechanism

```
1. Extract → validate output
2. If invalid: append error message to conversation → retry
   Error: "issue_date must be YYYY-MM-DD format, received: March 5th, 2026"
3. Repeat (max 3 attempts)
```

Specificity matters: the error must identify the field, what was received, and what was expected.
Vague errors do not converge.

### Diagnosis rule

Run 2-3 retries and compare the outputs for the failing field:
- **Same value** → Formatting Error → keep retrying or improve the error message
- **Different values** → Missing Information → stop and fix the schema

### Schema fix for Missing Information

```json
{
  "resolution_date": { "type": ["string", "null"], "description": "...or null if not yet resolved" },
  "agent_id":        { "type": ["string", "null"], "description": "...or null if not assigned" }
}
```

Remove the field from `required`. Set `type` to the nullable union. Add the system prompt null rule.

→ Full nullable schema patterns: [`null_handling_and_normalization.ipynb`](null_handling_and_normalization.ipynb)

---

### Terminology map

| This notebook | Standard AI engineering term | Notes |
|---|---|---|
| Formatting Error | **Format Error** | Value exists in source; format is wrong; detectable by regex / schema validation |
| Missing Information | **Semantic Error** (subtype 1) | Format-correct but fabricated; format validation cannot catch it |
| *(not covered here)* | **Semantic Error** (subtype 2: Misinterpretation) | Field exists but model misreads it; requires confidence scoring + human review |

The retry loop in this notebook is a defense against Format Errors and can surface Missing Information through divergence diagnosis. It cannot detect Misinterpretation — that requires a confidence score routing layer.

→ Full **Semantic Error** coverage (both subtypes, confidence scoring, Human Review Queue routing): [`semantic_error_detection.ipynb`](semantic_error_detection.ipynb)